In [ ]:
import sqlite3
import random
from typing import List, Dict
import logging
import sys
from pathlib import Path
from config import Settings
import json
from pydantic import BaseModel, ValidationError


# Add parent folder to search path to import from config.py
parent_dir = Path().resolve().parent
print(parent_dir)
sys.path.append(str(parent_dir))


SYSTEM_PROMPT = (
    "You explain concepts to people at different age levels. "
    "Adapt explanations to the user (e.g. age, education, job)."
)



/home/schmi/projects/explain2me


In [2]:

# --------------------------------------------------
# Fetch data from DB
# --------------------------------------------------


def fetch_training_data(db_path: str):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("""
        SELECT d.page_id, p.title, d.kind, d.content
        FROM definitions d
        JOIN pages p ON d.page_id = p.id
        WHERE d.kind IN ('simple', 'technical', 'kids')
    """)

    rows = cur.fetchall()
    conn.close()

    return rows

db_path = Settings.get_db_path()
test_fetch = fetch_training_data(db_path=db_path)

test_fetch[1:4]


[(19,
  'Analytics',
  'simple',
  '[{"heading": "Introduction", "paragraphs": ["Analytics is the systematic processing of data or statistics. This means discovering, studying and explaining important patterns in data.", "Analytics turns raw data into useful information for making better decisions. Analytics uses the application of statistics, computer programming, and operations research to gain information from meanings of data."]}]'),
 (20,
  'Average',
  'simple',
  '[{"heading": "Introduction", "paragraphs": ["An average is the \\"normal\\" number of a group of numbers made by mixing the group of numbers.", "In math, an average is called a mean. It can be found by adding the numbers, then dividing the answer by the amount of numbers there were. There are different kinds of mean, and other things that are sometimes thought of as \\"average\\" such as median or mode (statistics)."]}, {"heading": "Sport", "paragraphs": ["In some sports, such as cricket and baseball, averages are used

In [3]:

# --------------------------------------------------
# Audience generator based on kind
# --------------------------------------------------

def generate_user_prompt(title: str, kind: str) -> str:
    """
    Generate user request based on definition kind.
    """

    if kind == "kids":
        age = random.randint(6, 13)
        templates = [
            f"Can you explain to me what {title} is? I am {age} years old.",
            f"What is {title}? Please explain it for a {age}-year-old.",
            f"I'm {age}. Can you help me understand {title}?",
            f"I am in elementary school. What is {title}?",
            f"Can you explain {title} in very basic terms?"
        ]

    elif kind == "simple":
        ages = random.randint(12, 18)
        templates = [
            f"Explain {title} to a {ages}-year-old student.",
            f"What is {title}? I'm in high school.",
            f"Can you explain {title} in simple terms?"
        ]

    elif kind == "technical":
        roles = [
            "PhD student",
            "graduate student",
            "researcher",
            "engineer",
            "domain expert"
        ]
        role = random.choice(roles)

        templates = [
            f"What is {title}? Explain it to a {role}.",
            f"Provide a detailed explanation of {title} suitable for a {role}.",
            f"I am a {role}. Give me a technical explanation of {title}."
        ]

    else:
        templates = [f"Explain {title}."]

    return random.choice(templates)


test_gen_user_prompt = generate_user_prompt(title='correlation', kind='technical')
test_gen_user_prompt


'What is correlation? Explain it to a domain expert.'

In [ ]:

# --------------------------------------------------
# Cleaning function for classic Wikipedia page content
# --------------------------------------------------

class Section(BaseModel):
    heading: str
    paragraphs: List[str]


def clean_content(content: str) -> str:
    try:
        raw = json.loads(content)
        sections = [Section(**item) for item in raw]
    except (json.JSONDecodeError, ValidationError, TypeError) as e:
        raise ValueError("Input must be a str representation of a JSON list of objects with keys 'heading' and 'paragraphs' corresponding to str, resp. List[str]") from e

    output_parts = []

    for section in sections:
        output_parts.append(f"[SECTION: {section.heading}]\n")

        for paragraph in section.paragraphs:
            output_parts.append(f"{paragraph}\n")

        output_parts.append("\n")

    return "\n".join(output_parts)


# unit test
data = [
    {"heading": "cacac", "paragraphs": ["cacacaca", "ooooder"]}
]
json_str = json.dumps(data)

clean_content(json_str)

'[SECTION: cacac]\n\ncacacaca\n\nooooder\n\n\n'

In [ ]:

# --------------------------------------------------
# LoRA Adapter: training dataset builder
# --------------------------------------------------


def build_lora_training_dataset(db_path: str) -> List[Dict]:
    """
    Build a LoRA training dataset.
    Each row contains:
      - page_id
      - messages: list of system/user/assistant dicts
    """
    rows = fetch_training_data(db_path)

    training_data = []

    for page_id, title, kind, content in rows:
        try:
            # Generate user prompt
            user_prompt = generate_user_prompt(title, kind)

            # Clean content if not for kids
            if kind != 'kids':
                content = clean_content(content).strip()

            # Prepare assistant content
            assistant_content = f"[TITLE: {title}]\n\n{content}"

            # Prepare messages
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_content},
            ]

            training_data.append({
                "page_id": page_id,
                "messages": messages,
            })

        except Exception as e:
            logging.warning(f"Skipping page_id={page_id} due to error: {e}")
    
    return training_data


# ---- quick test ----

test_lora_training_data = build_lora_training_dataset(db_path=db_path)
test_lora_training_data[:2]


[{'page_id': 9,
  'messages': [{'role': 'system',
    'content': 'You explain concepts to people at different age levels. Adapt explanations to the user (e.g. age, education, job).'},
   {'role': 'user',
    'content': 'Provide a detailed explanation of Blockchain suitable for a graduate student.'},
   {'role': 'assistant',
    'content': '[TITLE: Blockchain]\n\n[SECTION: Introduction]\n\nA blockchain is a distributed ledger with growing lists of records ( blocks ) that are securely linked together via cryptographic hashes. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data (generally represented as a Merkle tree, where data nodes are represented by leaves). Since each block contains information about the previous block, they effectively form a chain ( viz. linked list data structure), with each additional block linking to the ones before it. Consequently, blockchain transactions are resistant to alteration because, once recorded, the data

In [ ]:
# --------------------------------------------------
# Store & Load LoRA training data
# --------------------------------------------------

training_data_path = "training_data.json"

def save_data(data, data_path):
    with open(data_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_data(data_path):
    with open(data_path,"r") as file:
        test_training_data = json.load(file)
    return test_training_data




save_data(data=test_lora_training_data,
          data_path=training_data_path)

test_training_data = load_data(data_path=training_data_path)
    
test_training_data[:2]

[{'page_id': 9,
  'messages': [{'role': 'system',
    'content': 'You explain concepts to people at different age levels. Adapt explanations to the user (e.g. age, education, job).'},
   {'role': 'user',
    'content': 'Provide a detailed explanation of Blockchain suitable for a graduate student.'},
   {'role': 'assistant',
    'content': '[TITLE: Blockchain]\n\n[SECTION: Introduction]\n\nA blockchain is a distributed ledger with growing lists of records ( blocks ) that are securely linked together via cryptographic hashes. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data (generally represented as a Merkle tree, where data nodes are represented by leaves). Since each block contains information about the previous block, they effectively form a chain ( viz. linked list data structure), with each additional block linking to the ones before it. Consequently, blockchain transactions are resistant to alteration because, once recorded, the data

In [ ]:

# from transformers import AutoModelForCausalLM, AutoTokenizer
# from transformers import PreTrainedTokenizerBase
# import torch

# # Set device
# device = "cuda" if torch.cuda.is_available() else "cpu"

# Settings.MAX_INPUT_TOKENS
# model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_id).to(
#     device
# )

# tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

In [ ]:
# toy_messages = [{"role": 'system', 'content': 'sys content 1'},
#                 {"role": 'user', 'content': 'sys content 1'},
#                 {"role": 'assistant', 'content': 'sys content 2'*5 + 'lets add additional words so that it  should at least exceed it'}]
# toy_messages


# trucated_tokens = tokenizer.apply_chat_template(
#     toy_messages,
#     tokenize=True,
#     add_generation_prompt=False,
#     max_length = Settings.MAX_INPUT_TOKENS,
#     truncation=True,
# )

# untruncated_tokens = tokenizer.apply_chat_template(
#     toy_messages,
#     tokenize=True,
#     add_generation_prompt=False,
# )

# print("Final token length:", len(untruncated_tokens['input_ids']))
# print(len(trucated_tokens['input_ids']))
# print("Max allowed:", Settings.MAX_INPUT_TOKENS)

